In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [ ]:
df = pd.read_csv('export_IA.csv', encoding='utf-8-sig')
print(f"Dimensions brutes : {df.shape}")
df.head()

In [ ]:
print(df['age_estim'].describe())
df['age_estim'].hist(bins=40, color='steelblue', edgecolor='white')
plt.title('Distribution de age_estim')
plt.xlabel('Âge estimé (ans)')
plt.ylabel('Nombre d\'arbres')
plt.tight_layout()
plt.show()

In [ ]:
FEATURES_NUM = ['haut_tot', 'haut_tronc', 'tronc_diam', 'clc_nbr_diag']
FEATURES_CAT = ['remarquable']
TARGET = 'age_estim'

df_work = df[FEATURES_NUM + FEATURES_CAT + [TARGET]].copy()
df_work = df_work.replace('N/A', np.nan).dropna()
df_work = df_work[df_work[TARGET] > 0]

print(f"Après nettoyage : {df_work.shape[0]} arbres")
print(f"Âge min : {df_work[TARGET].min():.0f} ans | max : {df_work[TARGET].max():.0f} ans | moyenne : {df_work[TARGET].mean():.1f} ans")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_work[TARGET].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title("Distribution de l'âge estimé")
axes[0].set_xlabel('Âge (ans)')
axes[0].set_ylabel("Nombre d'arbres")

axes[1].scatter(df_work['tronc_diam'], df_work[TARGET], alpha=0.2, s=5, color='steelblue')
axes[1].set_title('Diamètre du tronc vs âge estimé')
axes[1].set_xlabel('tronc_diam (m)')
axes[1].set_ylabel('age_estim (ans)')

plt.tight_layout()
plt.savefig('distribution_age.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Corrélation des variables numériques avec l'âge
corr = df_work[FEATURES_NUM + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)
corr.plot(kind='bar', color='steelblue', edgecolor='white', figsize=(8, 4))
plt.title('Corrélation des variables numériques avec age_estim')
plt.ylabel('Corrélation de Pearson')
plt.xticks(rotation=30, ha='right')
plt.axhline(0, color='black', linewidth=0.7)
plt.tight_layout()
plt.savefig('correlation_features.png', dpi=120, bbox_inches='tight')
plt.show()
print(corr)

In [ ]:
X = df_work[FEATURES_NUM + FEATURES_CAT]
y = df_work[TARGET]

# Découpage 60% train / 20% validation / 20% test
X_temp,  X_test,  y_temp,  y_test  = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val,   y_train, y_val   = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), FEATURES_CAT)
])

In [ ]:
models = {
    'Ridge'                   : Ridge(),
    'Random Forest Regressor' : RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting'       : GradientBoostingRegressor(random_state=42)
}

def eval_set(pipe, X_s, y_s):
    yp = pipe.predict(X_s)
    return {
        'mae'   : mean_absolute_error(y_s, yp),
        'rmse'  : root_mean_squared_error(y_s, yp),
        'r2'    : r2_score(y_s, yp),
        'y_pred': yp,
    }

results = {}
for name, reg in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', reg)])
    pipe.fit(X_train, y_train)
    tr = eval_set(pipe, X_train, y_train)
    va = eval_set(pipe, X_val,   y_val)
    results[name] = {'pipe': pipe, 'train': tr, 'val': va}
    print(f"{name:30s} | Train RMSE={tr['rmse']:.2f} R²={tr['r2']:.3f} | Val RMSE={va['rmse']:.2f} R²={va['r2']:.3f}")

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Modèle'     : name,
        'RMSE Train' : r['train']['rmse'],
        'R² Train'   : r['train']['r2'],
        'MAE Val'    : r['val']['mae'],
        'RMSE Val'   : r['val']['rmse'],
        'R² Val'     : r['val']['r2'],
    })
summary = pd.DataFrame(rows)
summary.style.format(precision=3).highlight_min(
    subset=['RMSE Val', 'MAE Val'], color='lightgreen'
).highlight_max(
    subset=['R² Val'], color='lightgreen'
)

In [ ]:
CLASSES = ['Jeune', 'Adulte', 'Vieux']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, r) in zip(axes, results.items()):
    yp = r['val']['y_pred']
    ax.scatter(y_val, yp, alpha=0.3, s=10, color='steelblue')
    lim = (0, max(y_val.max(), yp.max()) + 5)
    ax.plot(lim, lim, 'r--', linewidth=1)
    ax.set_xlabel('Âge réel (ans)')
    ax.set_ylabel('Âge prédit (ans)')
    ax.set_title(f"{name}\n[Val] R²={r['val']['r2']:.3f}  RMSE={r['val']['rmse']:.1f} ans")
plt.suptitle('Âge prédit vs réel – Validation set', fontsize=13)
plt.tight_layout()
plt.savefig('regression_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
best_name = min(results, key=lambda n: results[n]['val']['rmse'])
r = results[best_name]['val']
print(f"Meilleur modèle sur validation : {best_name}")
print(f"  MAE={r['mae']:.2f} ans | RMSE={r['rmse']:.2f} ans | R²={r['r2']:.3f}")

In [ ]:
param_grid = {
    'reg__n_estimators' : [100, 300],
    'reg__max_depth'    : [3, 5, 7],
    'reg__learning_rate': [0.05, 0.1],
    'reg__subsample'    : [0.8, 1.0],
}

pipe_gb = Pipeline([
    ('prep', preprocessor),
    ('reg', GradientBoostingRegressor(random_state=42))
])

grid_search = GridSearchCV(
    pipe_gb, param_grid,
    cv=5, scoring='neg_root_mean_squared_error',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)  # train set uniquement

print(f"Meilleurs hyperparamètres : {grid_search.best_params_}")
print(f"Meilleur score CV RMSE    : {-grid_search.best_score_:.2f} ans")

In [ ]:
best_pipe = grid_search.best_estimator_

for label, X_s, y_s in [('Train', X_train, y_train), ('Validation', X_val, y_val), ('Test', X_test, y_test)]:
    m = eval_set(best_pipe, X_s, y_s)
    print(f"[{label:10s}] MAE={m['mae']:.2f} ans  RMSE={m['rmse']:.2f} ans  R²={m['r2']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_pred_test = best_pipe.predict(X_test)
axes[0].scatter(y_test, y_pred_test, alpha=0.3, s=10, color='steelblue')
lim = (0, max(y_test.max(), y_pred_test.max()) + 5)
axes[0].plot(lim, lim, 'r--', linewidth=1)
axes[0].set_xlabel('Âge réel (ans)')
axes[0].set_ylabel('Âge prédit (ans)')
axes[0].set_title('Âge prédit vs réel – Test set\n'
                  f'MAE={mean_absolute_error(y_test, y_pred_test):.2f} ans  R²={r2_score(y_test, y_pred_test):.3f}')

gb_clf = best_pipe.named_steps['reg']
ohe_cols = best_pipe.named_steps['prep'].named_transformers_['cat'] \
    .get_feature_names_out(FEATURES_CAT)
all_features = FEATURES_NUM + list(ohe_cols)
importances = pd.Series(gb_clf.feature_importances_, index=all_features).sort_values(ascending=True)
importances.tail(12).plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Importance des variables (top 12)')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('gradient_boosting_resultats.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Sauvegarde du modèle

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(best_pipe, 'models/reg_pipeline.pkl')
joblib.dump({'features_num': FEATURES_NUM, 'features_cat': FEATURES_CAT}, 'models/meta.pkl')

print("Pipeline sauvegardé dans models/reg_pipeline.pkl")